# Logistic Regression Assumptions — Cybersecurity Practice Skeleton

**Short name (GitHub):** `LogReg_Cyber`  
**Lab source:** Codecademy *Assumptions of Logistic Regression I & II* adapted to a **SOC phishing-URL screen**.  
**Data:** `data/cyber_phishing.csv` (720 proxy events) and `data/cyber_logins.csv` (brute-force practice).

Work this notebook first. Use `LogReg_Cyber_Solution.ipynb` as the key. Helpers: `LogReg_Cyber.py`. Flowchart: `logreg_cyber_flowchart.png`.

> SOC framing: **missing a phishing URL (false negative) lets a credential-harvest page reach a user.** A false alarm only opens a ticket. Recall / catch-rate is the headline metric. Accuracy is secondary because benign URLs outnumber phishing (477 / 243).


## Inline cheat-sheet (keep this cell visible)

See also **`LogReg_Cyber_Cheatsheet.docx`**.

| Item | Formula / code |
|------|----------------|
| Encode target | `df['phishing'].map({'PHISH':1,'BENIGN':0})` |
| Binary check | `value_counts()` — exactly two classes |
| Independence | `event_id.nunique() == event_id.count()` |
| Events-per-variable | `max_features = min(class counts) / 10` |
| Logit linearity | `sns.regplot(..., logistic=True)` looks sigmoidal |
| Multicollinearity | drop one of a pair with \|r\| ≳ 0.8 (`url_length` vs `path_length`) |
| Unregularized LR | `LogisticRegression(penalty=None, fit_intercept=True)` |
| Soft scores | `predict_proba(X)[:, 1]` = P(phishing) |
| Hard class at *t* | `(proba >= t).astype(int)` |
| Recall (catch rate) | $\mathrm{TP}/(\mathrm{TP}+\mathrm{FN})$ |
| Precision (ticket purity) | $\mathrm{TP}/(\mathrm{TP}+\mathrm{FP})$ |
| ROC / AUC | `roc_curve` / `roc_auc_score` on **probabilities** |
| Stratify | `train_test_split(..., stratify=y)` |
| Balanced weights | `class_weight='balanced'` |

Default $t=0.5$ is a software convention, not a SOC policy. Lower *t* → more tickets, fewer misses.


## 0. Packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score,
    f1_score, roc_curve, roc_auc_score,
)
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler

%matplotlib inline
sns.set_style("whitegrid")
np.set_printoptions(precision=4, suppress=True)
print("libraries ready")


## 1. Load and encode

### Task 1.1
Read `data/cyber_phishing.csv`. Map `PHISH→1`, `BENIGN→0`. Print `head()` and `phishing.value_counts()`.


In [ ]:
# YOUR CODE HERE
df = None

print(df.head())
print(df.phishing.value_counts())


## 2. Assumptions I — target, independence, sample size, outliers

### Task 2.1 — binary target
Confirm exactly two classes. Phishing is the **positive / minority** class.


In [ ]:
# YOUR CODE HERE
print(df.phishing.value_counts())
print("n classes:", df.phishing.nunique())


### Task 2.2 — independent events
Each row is one URL request. Test `event_id.nunique() == event_id.count()`.
Repeated beacons from the same campaign would violate this — that is a later GEE / mixed-logit problem.


In [ ]:
# YOUR CODE HERE
unique_ids = None
print(unique_ids)


### Task 2.3 — events-per-variable cap
`max_features = min(class counts) / 10`. With 243 phishing events the cap is 24.3. Our 5-feature working set is inside it.


In [ ]:
# YOUR CODE HERE
max_features = None
print(max_features)


### Task 2.4 — outlier screen
Continuous features are right-skewed. Plot the log-then-z-score boxplot of `cont_features`. Which column has the longest upper tail?


In [ ]:
all_features = [
    "url_length", "path_length", "num_dots", "num_hyphens", "num_subdomains",
    "has_ip", "has_https", "has_at", "digit_ratio", "redirect_count", "request_hour",
]
cont_features = [
    "url_length", "path_length", "num_dots", "num_hyphens", "num_subdomains",
    "digit_ratio", "redirect_count", "request_hour",
]
# YOUR CODE HERE


### Task 2.5 — drop extreme `redirect_count`
Keep rows below the 99th percentile. Save as `df_filtered`. How many rows dropped?


In [ ]:
# YOUR CODE HERE
q_hi = None
df_filtered = None
print("q99 =", q_hi, " kept =", len(df_filtered), " dropped =", len(df) - len(df_filtered))


### Task 2.6 — redraw the boxplot on `df_filtered`


In [ ]:
# YOUR CODE HERE


## 3. Assumptions II — logit linearity and multicollinearity

### Task 3.1
Compare `sns.regplot(..., logistic=True)` for `url_length` (should look sigmoidal) and `request_hour` (flat / weak — clock hour is a poor linear logit feature).


In [ ]:
# YOUR CODE HERE


### Task 3.2 — correlation heatmap
`path_length` is almost a rescaling of `url_length`. Store that pair as `correlated_pair` and **do not** put both in the model.


In [ ]:
# YOUR CODE HERE
correlated_pair = []
print("correlated_pair:", correlated_pair)


## 4. scikit-learn implementation

Working set (drops the collinear path and the weak hour):

`core = ['url_length','num_subdomains','has_ip','has_https','digit_ratio']`

### Task 4.1 — construct the estimator
Unregularized, with intercept. Print `.get_params()`.


In [ ]:
core = ["url_length", "num_subdomains", "has_ip", "has_https", "digit_ratio"]
outcome = "phishing"
x_train, x_test, y_train, y_test = train_test_split(
    df[core], df[outcome], random_state=0, test_size=0.3
)
# YOUR CODE HERE
log_reg = None
print(log_reg.get_params())


### Task 4.2 — fit and read coefficients
`has_https` should come out **negative** (TLS is more common on legitimate sites in this file). `has_ip` and `digit_ratio` should be positive.


In [ ]:
# YOUR CODE HERE
coefficients = None
intercept = None
print("coefficients:", coefficients)
print("intercept:", intercept)


### Task 4.3 — test-set metrics
Which metric is highest / lowest? Why is accuracy a weak SOC headline here?


In [ ]:
# YOUR CODE HERE
y_pred = None
accuracy = precision = recall = f1 = None
print(accuracy, precision, recall, f1)


### Task 4.4 — confusion matrix
Rows = actual, columns = predicted. The dangerous cell is **FN** (actual phish, predicted benign).


In [ ]:
# YOUR CODE HERE
test_conf_matrix = None
print(test_conf_matrix)


## 5. Prediction thresholds

### Task 5.1
Rebuild the 0.5 class from `predict_proba` and confirm it matches `predict` with `np.array_equal`.


In [ ]:
y_pred_prob = log_reg.predict_proba(x_test)
# YOUR CODE HERE
y_pred_class = None
diff = None
print("same as predict()?", diff)


### Task 5.2 / 5.3 — CMs at 25%, 50%, 75%
As *t* rises, FN (missed phish) should rise and FP (extra tickets) should fall.


In [ ]:
# YOUR CODE HERE
print("CM 50%"); print(None)
print("CM 25%"); print(None)
print("CM 75%"); print(None)


### Task 5.4 — SOC threshold
Sweep `thresh = np.linspace(0, 1, 100)`. Record FN. What is the **lowest** threshold at which FN first reaches 8? Store as `thresh_choice`.


In [ ]:
thresh = np.linspace(0, 1, 100)
false_negatives = []
# YOUR CODE HERE
thresh_choice = None
print("thresh_choice =", thresh_choice)


## 6. ROC curve and AUC

### Task 6.1
Plot the model ROC and a `DummyClassifier(strategy='most_frequent')`. Label a few thresholds.


In [ ]:
# YOUR CODE HERE


### Task 6.2
`roc_auc` from `y_test` and the **positive-class probability**.


In [ ]:
# YOUR CODE HERE
roc_auc = None
print("ROC AUC:", roc_auc)


## 7. Class imbalance

Positivity rate $= n_{\text{phish}} / n \approx 243/720 \approx 0.34$.

### Task 7.1 — stratified split
`random_state=6`, `test_size=0.3`, `stratify=df[outcome]`.


In [ ]:
x_train_u, x_test_u, y_train_u, y_test_u = train_test_split(
    df[core], df[outcome], random_state=6, test_size=0.3
)
print("unstrat train pos", y_train_u.mean(), "test pos", y_test_u.mean())
# YOUR CODE HERE — x_train_str, x_test_str, y_train_str, y_test_str


### Task 7.2 — positivity rates after stratification


In [ ]:
# YOUR CODE HERE
str_train_positivity_rate = None
str_test_positivity_rate = None
print(str_train_positivity_rate, str_test_positivity_rate)


### Task 7.3 — recall / accuracy on the stratified fold


In [ ]:
# YOUR CODE HERE
recall_str = accuracy_str = None
print("stratified recall, acc:", recall_str, accuracy_str)


### Task 7.4 / 7.5 — `class_weight='balanced'`
Fit on the unstratified `random_state=6` split. Print recall and accuracy.


In [ ]:
# YOUR CODE HERE
log_reg_bal = None
recall_bal = accuracy_bal = None
print("balanced recall, acc:", recall_bal, accuracy_bal)


## 8. Alternate code

### Task 8.1 — statsmodels Logit
Compare signs with sklearn.


In [ ]:
# YOUR CODE HERE
# import statsmodels.api as sm


### Task 8.2 — scale-then-LR pipeline
Predictions stay close; coefficients switch to z-units (easier to compare `url_length` vs `digit_ratio`).


In [ ]:
# YOUR CODE HERE


### Task 8.3 — NumPy threshold helper


In [ ]:
def predict_at(proba, t=0.5):
    # YOUR CODE HERE
    pass

for t in (0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8):
    pred = predict_at(y_pred_prob[:, 1], t)
    cm = confusion_matrix(y_test, pred)
    print(t, "FN", cm[1, 0], "FP", cm[0, 1])


## 9. More practice

### Task 9.1 — add `has_at` and `num_dots`
Does recall or AUC move enough to justify the extra EPV spend?


In [ ]:
extra = core + ["has_at", "num_dots"]
# YOUR CODE HERE


### Task 9.2 — transfer problem: brute-force logins
`data/cyber_logins.csv` — `failed_logins`, `src_unique_ua`, `geo_rare`, `off_hours` → `brute_force`.

1. Positivity rate and 10-EPV cap.
2. Drop `packets` if it is collinear with `bytes_sent`.
3. Fit unregularized LR; print coefficients, recall, AUC.
4. Pick a threshold that keeps FN ≤ 10 on the test fold (or document that you cannot).


In [ ]:
logins = pd.read_csv("data/cyber_logins.csv")
# YOUR CODE HERE


### Task 9.3 — which metric when?
One sentence each: inbound email gateway, after-hours IR war-room, quarterly board pack.


In [ ]:
gateway = """..."""
war_room = """..."""
board = """..."""
print(gateway); print(war_room); print(board)


## 10. Simulation (edit the parameters)

Change `N`, `NOISE`, `T` and re-run. Questions:

* What happens to catch-rate if you raise `T` from 0.5 to 0.8?
* Does `class_weight='balanced'` still help at `N=120`?
* How much label noise (`NOISE=0.10`) does it take to drag AUC under 0.80?


In [ ]:
# --- editable parameters ---
N = 720
N_REPS = 20
NOISE = 0.00
T = 0.50
TEST_SIZE = 0.30
SEED = 0
# ---------------------------
rng = np.random.default_rng(SEED)
rows_def, rows_bal = [], []
pool_X = df[core].to_numpy()
pool_y = df[outcome].to_numpy()
for r in range(N_REPS):
    idx = rng.choice(len(df), size=N, replace=(N > len(df)))
    X, y = pool_X[idx], pool_y[idx]
    try:
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE, random_state=r, stratify=y)
    except ValueError:
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE, random_state=r)
    if NOISE > 0:
        flip = rng.random(len(ytr)) < NOISE
        ytr = ytr.copy(); ytr[flip] = 1 - ytr[flip]
    for tag, cw in (("default", None), ("balanced", "balanced")):
        m = LogisticRegression(penalty=None, fit_intercept=True, max_iter=4000, class_weight=cw)
        m.fit(Xtr, ytr)
        proba = m.predict_proba(Xte)[:, 1]
        pred = (proba >= T).astype(int)
        rec = dict(
            recall=recall_score(yte, pred, zero_division=0),
            accuracy=accuracy_score(yte, pred),
            auc=roc_auc_score(yte, proba) if len(np.unique(yte)) == 2 else np.nan,
        )
        (rows_def if tag == "default" else rows_bal).append(rec)
sim_def = pd.DataFrame(rows_def); sim_bal = pd.DataFrame(rows_bal)
print("DEFAULT\n", sim_def.agg(["mean", "std"]).round(3))
print("BALANCED\n", sim_bal.agg(["mean", "std"]).round(3))
fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
for ax, col in zip(axes, ["recall", "accuracy", "auc"]):
    ax.boxplot([sim_def[col].dropna(), sim_bal[col].dropna()], labels=["default", "balanced"])
    ax.set_title(col); ax.set_ylim(0.35, 1.02)
fig.suptitle(f"N={N}  T={T}  noise={NOISE}  reps={N_REPS}", y=1.03)
plt.tight_layout(); plt.show()


## 11. Audience rewrite

Using Jočys (data literacy, subject knowledge) and McMurrey (expert / technician / executive / nonspecialist), rewrite **one finding** four ways — e.g. “at t=0.25 we miss 7 phish and open 46 extra tickets; AUC ≈ 0.87”. ≤ 80 words each.


In [ ]:
expert = """..."""
technician = """..."""
executive = """..."""
nonspecialist = """..."""
print(expert); print(technician); print(executive); print(nonspecialist)


## 12. Top-10 applications — and when **not** to use this

10 cybersecurity settings where binary logit + assumption checks + threshold/ROC is the right tool, and 6–8 where it is the wrong tool.


In [ ]:
applications = []
not_appropriate = []
for row in applications: print(row)
print("--- not appropriate ---")
for row in not_appropriate: print(row)


## 13. Done-when checklist

- [ ] Encoded binary target, unique-id check, 10-EPV cap
- [ ] Outlier boxplots before / after the redirect filter
- [ ] Two logistic regplots and a labelled heatmap
- [ ] 5-feature unregularized model, 4 metrics, CM
- [ ] Threshold CMs + `thresh_choice`
- [ ] ROC vs dummy + AUC
- [ ] Stratify + `class_weight='balanced'`
- [ ] One alternate stack
- [ ] Extra-feature and brute-force practice
- [ ] Simulation that moves when you edit `N` / `T` / `NOISE`
- [ ] Four audience paragraphs + applications list

**Companions:** `LogReg_Cyber_Solution.ipynb`, `LogReg_Cyber_Reusable_Template.ipynb`, `LogReg_Cyber.py`, cheat sheet, 1-page report, memo, strategy guide, `logreg_cyber_flowchart.png`.
